# Momentum & Mean Reversion: Regime-Switching Strategy

## Summary

| | |
| :--- | :--- |
| **Idea** | Stay invested by default, but step out of a stock when it is overextended (RSI above 80 and more than 20% above its 50-day average) and return once it cools. |
| **Verdict** | Rejected |
| **Result** | With parameters tuned per ticker it beat buy and hold on some names (MSFT 1,163% vs 1,009%; BLDR, AMZN, GOOGL by more). It did not survive the robustness tests: it beat buy and hold over a trailing year on only 12% to 26% of days, and choosing parameters walk-forward gave -27.5% against buy and hold, ahead in 9.5% of months. |
| **Trust** | Medium for the rejection. The notebook attacks its own result with rolling, sensitivity, walk-forward and plateau tests, and the idea fails them. Caveats: only the 14 stocks held, no costs (which would make it worse), and an RSI variant that differs from the standard one. |
| **Date** | 2026-07-19 |
| **Code status** | Frozen: written before the current modules and no longer runs. Read it as a report. |

*Summary added 2026-10-13 by Claude from this notebook's own commentary and stored results. The verdict is a draft for Ben to confirm or correct.*

# Momentum & Mean Reversion: Regime-Switching Strategy
This notebook implements a dynamic strategy that enters stocks during early-stage momentum and exits them when they show signs of exhaustion (mean reversion).

### Strategy Lifecycle:
1.  **Entry Strategy**: Identify when a stock establishes a new upward trend (e.g., Price > 50-day SMA > 200-day SMA).
2.  **Exit Strategy**: Identify "overextension" using RSI and Distance-from-Mean to exit before momentum breaks down.
3.  **Backtest**: Evaluate this active switching against a simple "Buy and Hold" benchmark.

In [20]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Reuse our foundations
from PriceData import PriceData
from PriceIndicators import PriceIndicators

price_loader = PriceData("YF/yf-prices.csv")
indicators = PriceIndicators()

# Load specific watchlist
watchlist = pd.read_csv('degiro.csv')
tickers = watchlist['ticker'].tolist()

# Load pricing data (Wide format)
prices = price_loader.get_data_wide(tickers=tickers)
print(f"Loaded {len(tickers)} tickers. Analysis ready.")

Loaded 14 tickers. Analysis ready.


## 1. Tactical Buy & Hold Overlay
Instead of trying to catch a trend (which often fails due to lagging indicators), we will assume a **Buy & Hold** default. We only exit if "Mean Reversion Risk" becomes extreme.
*   **Default**: Stay Invested (Signal = 1).
*   **Tactical Exit**: RSI > 80 AND Price > 20% extension from 50 SMA.
*   **Re-entry**: Wait for a "Cool Down" (RSI < 50 or Price < 50 SMA).

In [25]:
def calculate_tactical_signals(prices, ticker):
    t_prices = prices[ticker].dropna()
    
    # 1. Technical Anchors
    sma_50 = t_prices.rolling(50).mean()
    sma_200 = t_prices.rolling(200).mean() # Still useful for context
    
    # 2. Indicators
    dist_from_sma50 = (t_prices / sma_50) - 1
    rsi = indicators.rsi(pd.DataFrame(t_prices), window=14)[ticker]
    
    # Logic for Tactical Overlay (State Machine)
    # 1 = Invested, 0 = Trimmed (Cash)
    signals = pd.Series(index=t_prices.index, data=1) 
    
    is_invested = True
    
    for i in range(1, len(t_prices)):
        today = t_prices.index[i]
        
        # EXIT LOGIC (Extreme Mean Reversion Risk)
        # Sell if RSI is extremely high AND price is disconnected from MA
        if is_invested:
            if rsi.iloc[i] > 80 and dist_from_sma50.iloc[i] > 0.20:
                is_invested = False
        
        # RE-ENTRY LOGIC (Cooled down)
        # Get back in if RSI falls back to neutral or price hits the 50 SMA
        else:
            if rsi.iloc[i] < 50 or t_prices.iloc[i] < sma_50.iloc[i]:
                is_invested = True
        
        signals.loc[today] = 1 if is_invested else 0
        
    return signals, sma_50, sma_200, rsi

# Test on MSFT
ticker_to_test = 'MSFT'
regime_signals, ma50, ma200, rsi_val = calculate_tactical_signals(prices, ticker_to_test)
print(f"Tactical signals calculated for {ticker_to_test}.")

Tactical signals calculated for MSFT.


## 2. Visualizing Tactical Trims
The green dots now represent when we are "In" the market. We expect to see 100% coverage most of the time, with small gaps during parabolic blow-off tops.

In [26]:
p = prices[ticker_to_test].tail(500)
s = regime_signals.tail(500)

fig = make_subplots(rows=2, cols=1, shared_xaxes=True, 
                    vertical_spacing=0.05, row_heights=[0.7, 0.3])

# Price Chart
fig.add_trace(go.Scatter(x=p.index, y=p, name='Price', line=dict(color='black', width=1)), row=1, col=1)
fig.add_trace(go.Scatter(x=p.index, y=ma50.tail(500), name='50 SMA', line=dict(dash='dot', color='blue')), row=1, col=1)

# Highlight "Active" periods in Green
active_segments = s == 1
fig.add_trace(go.Scatter(x=p.index[active_segments], y=p[active_segments], 
                         mode='markers', name='Active Trend', 
                         marker=dict(color='green', size=4, opacity=0.5)), row=1, col=1)

# RSI Chart
fig.add_trace(go.Scatter(x=p.index, y=rsi_val.tail(500), name='RSI', line=dict(color='purple')), row=2, col=1)
fig.add_hline(y=70, line_dash="dash", line_color="red", row=2, col=1)
fig.add_hline(y=30, line_dash="dash", line_color="green", row=2, col=1)

fig.update_layout(title=f'Regime Analysis: {ticker_to_test}', height=700, template='plotly_white')
fig.show()

## 3. Backtesting: Regime-Switching vs. Buy & Hold
We'll now calculate the cumulative returns for two scenarios:
1.  **Buy & Hold**: Simply holding the ticker for the entire period.
2.  **Regime Strategy**: Only holding the ticker on days where our "Active Trend" signal is 1, and being in "Cash" (0% return) on other days.


In [27]:
# Calculate daily returns for the ticker
daily_returns = prices[ticker_to_test].pct_change()

# strategy_returns: 
# If signal today is 1, our return for TOMORROW is whatever the stock does.
# We shift the signal by 1 day because you act on today's close for tomorrow's performance.
strategy_returns = daily_returns * regime_signals.shift(1)

# Calculate Cumulative Performance
cum_bh = (1 + daily_returns.fillna(0)).cumprod()
cum_strategy = (1 + strategy_returns.fillna(0)).cumprod()

# Plot comparison
fig_bt = go.Figure()
fig_bt.add_trace(go.Scatter(x=cum_bh.index, y=cum_bh, name='Buy & Hold'))
fig_bt.add_trace(go.Scatter(x=cum_strategy.index, y=cum_strategy, name='Regime Momentum Strategy'))

fig_bt.update_layout(title=f'Backtest: Strategy vs Buy & Hold ({ticker_to_test})',
                    yaxis_title='Cumulative Growth (1.0 = Base)',
                    template='plotly_white')
fig_bt.show()

# Final Performance stats
total_ret_bh = (cum_bh.iloc[-1] - 1) * 100
total_ret_strat = (cum_strategy.iloc[-1] - 1) * 100
print(f"Total Return Buy & Hold: {total_ret_bh:.2f}%")
print(f"Total Return Strategy: {total_ret_strat:.2f}%")


Total Return Buy & Hold: 1008.72%
Total Return Strategy: 1162.78%


## 4. Batch Analysis: Identifying "Strategy-Friendly" Tickers
We'll now run the Tactical Overlay performance across the entire watchlist. Our goal is to find tickers where the **Strategy Alpha** (Return of Strategy - Return of B&H) is positive, indicating that the stock's volatility is "predictably mean-reverting" rather than just a straight line up.

In [29]:
results = []

for ticker in tickers:
    try:
        # Calculate Signals
        sigs, _, _, _ = calculate_tactical_signals(prices, ticker)
        
        # Calculate Returns
        d_rets = prices[ticker].pct_change()
        s_rets = d_rets * sigs.shift(1)
        
        # Cumulative Performance
        c_bh = (1 + d_rets.fillna(0)).cumprod().iloc[-1]
        c_strat = (1 + s_rets.fillna(0)).cumprod().iloc[-1]
        
        # Count the number of unique "Offline" sessions
        # (Where signal changed from 1 to 0)
        trim_count = (sigs.diff() == -1).sum()
        
        results.append({
            'Ticker': ticker,
            'B&H Ret %': (c_bh - 1) * 100,
            'Strat Ret %': (c_strat - 1) * 100,
            'Alpha %': (c_strat - c_bh) * 100,
            'Trims': trim_count
        })
    except Exception as e:
        continue

# Convert to DataFrame and Rank
batch_results = pd.DataFrame(results).sort_values(by='Alpha %', ascending=False)

# Filter for meaningful tickers
# Tiers: Outperformers vs Underperformers
print("Top Tickers where Strategy added value:")
print(batch_results[batch_results['Alpha %'] > 0].to_string(index=False))

# Visualize
fig_alpha = go.Figure()
colors = ['green' if x > 0 else 'red' for x in batch_results['Alpha %']]
fig_alpha.add_trace(go.Bar(x=batch_results['Ticker'], y=batch_results['Alpha %'], 
                           marker_color=colors))
fig_alpha.update_layout(title='Tactical Strategy Alpha (All Tickers)', 
                        yaxis_title='Outperformance vs B&H (%)',
                        template='plotly_white')
fig_alpha.show()

Top Tickers where Strategy added value:
Ticker    B&H Ret %  Strat Ret %     Alpha %  Trims
  BLDR   452.189772  1898.909721 1446.719949     23
  AMZN  5432.419485  6830.851416 1398.431931     15
 GOOGL 13824.115030 15187.598930 1363.483900     11
  ULTA  1526.211051  1997.618834  471.407783      7
  MSFT  1008.715815  1162.780670  154.064855      3
  UBER    74.308395   163.547679   89.239285      9


## 5. Challenging the Strategy: Rolling Alpha Analysis
To ensure the outperformance isn't just luck or a single well-timed trade, we will calculate the **Rolling 1-Year Alpha**. This shows the strategy's performance minus Buy & Hold performance over every trailing 252-day window.
*   **Above 0**: The strategy was adding value over that year.
*   **Below 0**: Buy & Hold was the better choice during that period.

In [30]:
def plot_rolling_alpha(prices, ticker, window=252):
    # 1. Calculate returns and strategy signals
    sigs, _, _, _ = calculate_tactical_signals(prices, ticker)
    d_rets = prices[ticker].pct_change().fillna(0)
    s_rets = (d_rets * sigs.shift(1)).fillna(0)
    
    # 2. Calculate rolling cumulative returns for both
    # We use a rolling window to see if the outperformance is consistent
    rolling_bh = (1 + d_rets).rolling(window).apply(np.prod, raw=True) - 1
    rolling_strat = (1 + s_rets).rolling(window).apply(np.prod, raw=True) - 1
    
    rolling_alpha = (rolling_strat - rolling_bh) * 100
    
    # 3. Visualize
    fig_roll = go.Figure()
    fig_roll.add_trace(go.Scatter(x=rolling_alpha.index, y=rolling_alpha, 
                                 name='Rolling 1-Year Alpha',
                                 line=dict(color='blue')))
    fig_roll.add_hline(y=0, line_dash="dash", line_color="black")
    
    fig_roll.update_layout(title=f'Rolling 1-Year Alpha: {ticker} (Strategy vs B&H)',
                          yaxis_title='Outperformance (%)',
                          template='plotly_white')
    fig_roll.show()
    
    # Consistency Metric: % of time Alpha is positive
    consistency = (rolling_alpha > 0).mean() * 100
    print(f"Consistency Score (Percent of days outperforming B&H over trailing year): {consistency:.1f}%")

# Test on a "Winner" like MSFT or AMZN
plot_rolling_alpha(prices, 'MSFT')
plot_rolling_alpha(prices, 'AMZN')

Consistency Score (Percent of days outperforming B&H over trailing year): 11.6%


Consistency Score (Percent of days outperforming B&H over trailing year): 26.1%


## 6. Sensitivity Analysis: Parameter Heatmap
To determine if our strategy is robust or just overfitted to specific numbers, we'll test a range of:
*   **RSI Thresholds** (e.g., 70 to 90)
*   **Extension Thresholds** (e.g., 10% to 35% above the 50 SMA)

A **Robust** strategy will show a broad "plateau" of green (positive Alpha) in the heatmap. A **Fragile** strategy will show only isolated spots of success.

In [33]:
def get_alpha_for_params(ticker_prices, rsi_series, dist_series, rsi_thresh, dist_thresh):
    # vectorized logic for speed in heatmap
    signals = pd.Series(index=ticker_prices.index, data=1)
    is_invested = True
    
    # We still need a loop because it's a state machine (re-entry depends on prev state)
    for i in range(1, len(ticker_prices)):
        if is_invested:
            if rsi_series.iloc[i] > rsi_thresh and dist_series.iloc[i] > dist_thresh:
                is_invested = False
        else:
            if rsi_series.iloc[i] < 50 or dist_series.iloc[i] < 0:
                is_invested = True
        signals.iloc[i] = 1 if is_invested else 0
        
    daily_rets = ticker_prices.pct_change().fillna(0)
    strat_rets = daily_rets * signals.shift(1).fillna(1)
    
    total_bh = (1 + daily_rets).prod() - 1
    total_strat = (1 + strat_rets).prod() - 1
    return (total_strat - total_bh) * 100

def plot_parameter_heatmap(prices, ticker):
    t_prices = prices[ticker].dropna()
    sma50 = t_prices.rolling(50).mean()
    dist_from_sma50 = (t_prices / sma50) - 1
    rsi = indicators.rsi(pd.DataFrame(t_prices), window=14)[ticker]
    
    rsi_range = np.arange(70, 91, 2) # RSI from 70 to 90
    dist_range = np.arange(0.02, 0.36, 0.02) # Extension from 2% to 35%
    
    heatmap_data = np.zeros((len(rsi_range), len(dist_range)))
    
    for r_idx, r_thresh in enumerate(rsi_range):
        for d_idx, d_thresh in enumerate(dist_range):
            heatmap_data[r_idx, d_idx] = get_alpha_for_params(t_prices, rsi, dist_from_sma50, r_thresh, d_thresh)
            
    fig_heat = go.Figure(data=go.Heatmap(
        z=heatmap_data,
        x=[f"{d*100:.0f}%" for d in dist_range],
        y=rsi_range,
        colorscale=[
            [0, 'rgb(165,0,38)'],   # Strong Red
            [0.45, 'rgb(255,150,150)'], # Soft Red (close to 0)
            [0.5, 'rgb(255,255,255)'], # Pure White (0.0)
            [0.55, 'rgb(150,255,150)'], # Soft Green (just above 0)
            [1, 'rgb(0,104,55)']    # Strong Green
        ],
        zmid=0,
        colorbar=dict(title='Alpha %'),
        text=np.round(heatmap_data, 1),
        texttemplate="%{text}%",
        textfont={"size":10}
    ))
    
    fig_heat.update_layout(
        title=f'Sensitivity Analysis: {ticker} Strategy Alpha',
        xaxis_title='Extension Threshold (% above 50 SMA)',
        yaxis_title='RSI Threshold',
        template='plotly_white'
    )
    fig_heat.show()

# Run for our top winner
plot_parameter_heatmap(prices, 'MSFT')

## 7. Parameter Stability Analysis
To distinguish between "luck" and a "reliable pattern," we will track the **Top 3 performing cells** of the heatmap over time.
1.  We'll use a **rolling 12-month window**.
2.  Every month, we'll re-calculate the Alpha for all parameter combinations.
3.  We'll "vote" for the top 3 cells.
4.  The final heatmap will show which parameters consistently appeared as "winners."

**Concentrated clusters** = Reliable pattern.
**Scattered dots** = Random outcomes.

In [ ]:
def plot_parameter_stability(prices, ticker):
    t_prices = prices[ticker].dropna()
    sma50 = t_prices.rolling(50).mean()
    dist_from_sma50 = (t_prices / sma50) - 1
    rsi = indicators.rsi(pd.DataFrame(t_prices), window=14)[ticker]
    
    # Define Narrower, more focused ranges
    rsi_range = np.arange(60, 86, 2)
    dist_range = np.arange(0.06, 0.21, 0.02)
    
    # Grid of parameter combinations
    param_grid = []
    for r in rsi_range:
        for d in dist_range:
            param_grid.append((r, d))
            
    # Track "Top 3 frequency"
    votes = np.zeros((len(rsi_range), len(dist_range)))
    
    # Rolling 12-month windows, shifting by 1 month (approx 21 trading days)
    window_days = 252
    step_days = 21
    
    print(f"Analyzing {ticker} parameter stability...")
    
    for start_idx in range(0, len(t_prices) - window_days, step_days):
        end_idx = start_idx + window_days
        
        subset_prices = t_prices.iloc[start_idx:end_idx]
        subset_rsi = rsi.iloc[start_idx:end_idx]
        subset_dist = dist_from_sma50.iloc[start_idx:end_idx]
        
        window_alphas = []
        for r_thresh, d_thresh in param_grid:
            alpha = get_alpha_for_params(subset_prices, subset_rsi, subset_dist, r_thresh, d_thresh)
            window_alphas.append(alpha)
            
        # Find Top 3 indices
        top_3_indices = np.argsort(window_alphas)[-3:]
        
        for idx in top_3_indices:
            r_val, d_val = param_grid[idx]
            r_pos = np.where(rsi_range == r_val)[0][0]
            d_pos = np.where(dist_range == d_val)[0][0]
            votes[r_pos, d_pos] += 1
            
    # Visualize Stability
    fig_stable = go.Figure(data=go.Heatmap(
        z=votes,
        x=[f"{d*100:.0f}%" for d in dist_range],
        y=rsi_range,
        colorscale='Greens',
        colorbar=dict(title='Top 3 Frequency')
    ))
    
    fig_stable.update_layout(
        title=f'Parameter Stability: {ticker} (Top 3 Performance Frequency)',
        xaxis_title='Extension Threshold (% above 50 SMA)',
        yaxis_title='RSI Threshold',
        template='plotly_white'
    )
    fig_stable.show()

# Run for MSFT, BLDR, and NVDA to compare stability - very slow so don't run for lots of tickers
plot_parameter_stability(prices, 'MSFT')
# plot_parameter_stability(prices, 'BLDR')
# plot_parameter_stability(prices, 'NVDA')

Analyzing MSFT parameter stability...


Analyzing BLDR parameter stability...


Analyzing NVDA parameter stability...


## 8. Walk-Forward Proximity: Testing "Parameter Chase"
To see if a Walk-Forward approach actually works, we need to know if last year's **Optimal Parameters** are still "good enough" for next year.
1.  **Walk-Forward Execution**: Every month, find the best parameters from the *previous* 12 months.
2.  **The Proximity Test**: Calculate how far the *new* optimal parameters are from the *old* ones on the heatmap.
3.  **Real-World Returns**: Execute the strategy using only the "lagged" optimal parameters to see if we still beat Buy & Hold.

If the "Optimal Zone" moves slowly and predictably, the strategy has an **Edge**. If it jumps randomly, we are just chasing ghosts.

In [36]:
def run_walk_forward_test(prices, ticker):
    t_prices = prices[ticker].dropna()
    sma50 = t_prices.rolling(50).mean()
    dist_from_sma50 = (t_prices / sma50) - 1
    rsi = indicators.rsi(pd.DataFrame(t_prices), window=14)[ticker]
    
    rsi_range = np.arange(60, 86, 2)
    dist_range = np.arange(0.06, 0.21, 0.02)
    param_grid = [(r, d) for r in rsi_range for d in dist_range]
    
    window_days = 252 # 1 year lookback
    step_days = 21   # Check once a month
    
    wf_results = []
    
    # Start after we have enough data for the first lookback
    for current_idx in range(window_days, len(t_prices) - step_days, step_days):
        # 1. Look back: What was optimal in the LAST 12 months?
        lookback_start = current_idx - window_days
        lookback_end = current_idx
        
        lb_prices = t_prices.iloc[lookback_start:lookback_end]
        lb_rsi = rsi.iloc[lookback_start:lookback_end]
        lb_dist = dist_from_sma50.iloc[lookback_start:lookback_end]
        
        best_alpha = -np.inf
        best_params = (None, None)
        
        for r_th, d_th in param_grid:
            a = get_alpha_for_params(lb_prices, lb_rsi, lb_dist, r_th, d_th)
            if a > best_alpha:
                best_alpha = a
                best_params = (r_th, d_th)
        
        # 2. Look forward: How did those parameters do in the NEXT month?
        lf_start = current_idx
        lf_end = current_idx + step_days
        
        lf_prices = t_prices.iloc[lf_start:lf_end]
        lf_rsi = rsi.iloc[lf_start:lf_end]
        lf_dist = dist_from_sma50.iloc[lf_start:lf_end]
        
        # Execute strategy for the next month using the SLIGHTLY LAGGED best params
        actual_alpha = get_alpha_for_params(lf_prices, lf_rsi, lf_dist, best_params[0], best_params[1])
        
        wf_results.append({
            'Date': t_prices.index[current_idx],
            'Best RSI': best_params[0],
            'Best Dist': best_params[1],
            'Monthly Alpha %': actual_alpha
        })
        
    wf_df = pd.DataFrame(wf_results)
    
    # Visualize parameter drift
    fig_drift = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.1,
                              subplot_titles=("Optimal RSI Drift", "Optimal Extension Drift"))
    
    fig_drift.add_trace(go.Scatter(x=wf_df['Date'], y=wf_df['Best RSI'], name='RSI'), row=1, col=1)
    fig_drift.add_trace(go.Scatter(x=wf_df['Date'], y=wf_df['Best Dist']*100, name='Extension %'), row=2, col=1)
    
    fig_drift.update_layout(height=600, title=f"Walk-Forward Parameter Stability: {ticker}", template='plotly_white')
    fig_drift.show()
    
    cum_alpha = wf_df['Monthly Alpha %'].sum()
    print(f"Total Alpha generated by Walk-Forward Strategy: {cum_alpha:.2f}%")
    print(f"Percentage of months outperforming B&H: {(wf_df['Monthly Alpha %'] > 0).mean()*100:.1f}%")

run_walk_forward_test(prices, 'MSFT')

Total Alpha generated by Walk-Forward Strategy: -27.52%
Percentage of months outperforming B&H: 9.5%


## 9. Global Robustness: Finding the "Universal" Rule
If a strategy is truly an "Edge," it shouldn't require surgical precision for every individual ticker. We will now calculate the **Average Alpha across the entire watchlist** for each parameter combination.

**This is the ultimate test**: If there is no "Universal Zone" where the majority of stocks benefit, then the strategy is likely just curve-fitting the past. If there is a "Global Green Zone," then we have a systematic rule for trimming parabolic risk.

In [38]:
def plot_global_parameter_heatmap(prices, tickers):
    # Standard grid
    rsi_range = np.arange(70, 91, 2)
    dist_range = np.arange(0.06, 0.26, 0.02)
    
    # Pre-calculate data for efficiency
    indicators_data = {}
    valid_tickers = []
    
    print(f"Pre-calculating indicators for {len(tickers)} tickers...")
    for ticker in tickers:
        try:
            t_data = prices[ticker].dropna()
            if len(t_data) < 500: continue 
            
            sma50 = t_data.rolling(50).mean()
            dist = (t_data / sma50) - 1
            rsi = indicators.rsi(pd.DataFrame(t_data), window=14)[ticker]
            indicators_data[ticker] = (t_data, rsi, dist)
            valid_tickers.append(ticker)
        except:
            continue

    num_valid = len(valid_tickers)
    print(f"Analyzing {num_valid} valid tickers...")
    
    # We will use MEDIAN Alpha to prevent outliers (like NVDA) from breaking the scale
    median_alphas = np.zeros((len(rsi_range), len(dist_range)))
    win_rates = np.zeros((len(rsi_range), len(dist_range)))
    
    for r_idx, r_th in enumerate(rsi_range):
        for d_idx, d_th in enumerate(dist_range):
            ticker_alphas = []
            for ticker in valid_tickers:
                p, r, d = indicators_data[ticker]
                a = get_alpha_for_params(p, r, d, r_th, d_th)
                ticker_alphas.append(a)
            
            median_alphas[r_idx, d_idx] = np.median(ticker_alphas)
            win_rates[r_idx, d_idx] = (np.array(ticker_alphas) > 0).mean() * 100

    # 3. Legend/Scale Fix: 
    # Even median can be high, so we cap the COLOR scale at +/- 100% 
    # so we can actually see the variation in the "workable" range.
    display_z = np.clip(median_alphas, -100, 100)

    fig_global = go.Figure(data=go.Heatmap(
        z=display_z,
        x=[f"{d*100:.0f}%" for d in dist_range],
        y=rsi_range,
        colorscale='RdYlGn',
        zmid=0,
        text=np.round(median_alphas, 0), # Show the actual median alpha
        texttemplate="%{text}", # No "Win", no "%", just the number
        colorbar=dict(title='Median Alpha % (Capped at +/- 100)'),
        hoverongaps=False,
        hovertemplate="RSI: %{y}<br>Dist: %{x}<br>Median Alpha: %{z:.1f}%<br>Win Rate: %{customdata:.0f}%<extra></extra>",
        customdata=win_rates
    ))
    
    fig_global.update_layout(
        title=f'Global Portfolio Robustness (Median Alpha across {num_valid} tickers)',
        xaxis_title='Universal Extension Threshold (% above 50 SMA)',
        yaxis_title='Universal RSI Threshold',
        template='plotly_white',
    )
    fig_global.show()

plot_global_parameter_heatmap(prices, tickers)

Pre-calculating indicators for 14 tickers...
Analyzing 14 valid tickers...


## 10. The "Blur Test": Filtering for Plateaus vs. Peaks
To find a genuinely applicable strategy, we need to eliminate "Jagged Peaks" (lucky parameters). We will apply a **3x3 Neighborhood Average** to the heatmap values. 
*   **Survives the blur**: A robust "Goldilocks Zone" where parameters can drift slightly without the strategy breaking.
*   **Disappears**: A lucky fluke that was isolated and fragile.

If the result is mostly red or neutral, it confirms your skepticism: the strategy is just chasing noise.

In [39]:
from scipy.ndimage import uniform_filter

def plot_robust_global_heatmap(prices, tickers):
    # Standard grid (consistent with previous test)
    rsi_range = np.arange(70, 91, 2)
    dist_range = np.arange(0.06, 0.26, 0.02)
    
    # Pre-calculate indicators (we reuse the logic to ensure data is fresh)
    indicators_data = {}
    valid_tickers = []
    for ticker in tickers:
        try:
            t_data = prices[ticker].dropna()
            if len(t_data) < 500: continue 
            sma50 = t_data.rolling(50).mean()
            dist = (t_data / sma50) - 1
            rsi = indicators.rsi(pd.DataFrame(t_data), window=14)[ticker]
            indicators_data[ticker] = (t_data, rsi, dist)
            valid_tickers.append(ticker)
        except: continue

    raw_alphas = np.zeros((len(rsi_range), len(dist_range)))
    for r_idx, r_th in enumerate(rsi_range):
        for d_idx, d_th in enumerate(dist_range):
            alphas = [get_alpha_for_params(p, r, d, r_th, d_th) for ticker in valid_tickers for p, r, d in [indicators_data[ticker]]]
            raw_alphas[r_idx, d_idx] = np.median(alphas)

    # THE BLUR: 3x3 average filter
    # This specifically addresses your concern about isolated "lucky" cells
    robust_alphas = uniform_filter(raw_alphas, size=3, mode='constant', cval=0)

    fig_robust = go.Figure(data=go.Heatmap(
        z=np.clip(robust_alphas, -50, 50),
        x=[f"{d*100:.0f}%" for d in dist_range],
        y=rsi_range,
        colorscale='RdYlGn',
        zmid=0,
        text=np.round(robust_alphas, 1),
        texttemplate="%{text}",
        colorbar=dict(title='Blurred Median Alpha %'),
    ))
    
    fig_robust.update_layout(
        title=f'Robustness Filter: Blurred Global Alpha (3x3 neighborhood avg)',
        xaxis_title='Universal Extension Threshold',
        yaxis_title='Universal RSI Threshold',
        template='plotly_white'
    )
    fig_robust.show()

# Run the final check for a stable "Plateau"
plot_robust_global_heatmap(prices, tickers)